# 04 · Experimentos GRU con patching

Tres corridas (PPG, ECG, PPG+ECG) con **el mismo presupuesto**: AdamW(lr=1e-3, wd=1e-2), warmup de 1 época + coseno por lote,
batch 128, hasta 30 épocas, early stopping (paciencia 5) sobre `val_loss`, `clip_grad_norm_(1.0)`, semilla 0 antes de construir cada modelo.
Se registra en MLflow (`mlflow.db`). Arquitectura: ventanas de 25 muestras (T=150) → `GRU(64, 2 capas)` → `Linear(64, 2)` sobre el último estado oculto.
**Solo train y val**; test se evalúa una vez en el notebook 06.

In [1]:
import sys, json, time, warnings
from pathlib import Path
warnings.filterwarnings("ignore")
ROOT = Path.cwd().resolve()
while not (ROOT / "src" / "bp").exists():
    ROOT = ROOT.parent
sys.path.insert(0, str(ROOT / "src"))
import numpy as np, torch
print("ROOT:", ROOT, "| cuda:", torch.cuda.is_available())

ROOT: C:\Users\USUARIO\Documents\Source\dl_bp_mimic_proyecto_integrador | cuda: True


In [2]:
from bp.experiment import get_data, load_label_stats, save_val_result
from bp.train import fit
from bp.dataset import INPUT_SETS
import matplotlib; matplotlib.use("Agg")
import matplotlib.pyplot as plt
y_mean, y_std = load_label_stats()
runs = {}

In [3]:
variants = [('', {})]   # (sufijo, hiperparámetros del modelo); se elige por val_loss en el notebook 06
for suffix, model_kw in variants:
    for name, inputs in INPUT_SETS.items():
        tr, va = get_data('train', inputs), get_data('val', inputs)
        res = fit('build_gru', inputs, tr, va, y_mean, y_std, run_name='gru' + suffix + '_' + name, model_kw=model_kw)
        save_val_result(res); runs[suffix + name] = res
        print(f"==> gru{suffix} {name}: mejor época {res['best_epoch']} val_loss {res['best_val_loss']:.4f} "
              f"MAE val PAS/PAD {res['val_mae_pas']:.2f}/{res['val_mae_pad']:.2f} mmHg ({res['seconds']:.0f}s)")
        del tr, va; torch.cuda.empty_cache()

2026/11/04 01:31:47 INFO mlflow.agent.hint: Load the `mlflow-trace-instrumentation` skill at C:\Users\USUARIO\.conda\envs\dl_bp\Lib\site-packages\mlflow\assistant\skills\mlflow-trace-instrumentation\SKILL.md before writing any tracing code; it ships with this MLflow install. Set MLFLOW_DISABLE_AGENT_HINT=1 to silence this.


[gru_ppg] ep 00 train 0.9895 val 0.8806 MAE 13.97/9.00 gn 0.24


[gru_ppg] ep 01 train 0.9450 val 0.8729 MAE 13.93/9.07 gn 0.60


[gru_ppg] ep 02 train 0.9297 val 0.9061 MAE 14.21/9.47 gn 0.67


[gru_ppg] ep 03 train 0.9223 val 0.8523 MAE 13.65/8.75 gn 0.74


[gru_ppg] ep 04 train 0.9084 val 0.8472 MAE 13.65/8.77 gn 0.94


... (31 outputs left out)

[gru_ppg+ecg] ep 08 train 0.7218 val 0.8683 MAE 13.40/8.83 gn 1.78


[gru_ppg+ecg] ep 09 train 0.6967 val 0.8970 MAE 13.66/9.03 gn 1.66


[gru_ppg+ecg] ep 10 train 0.6738 val 0.8940 MAE 13.57/9.11 gn 1.81


==> gru ppg+ecg: mejor época 5 val_loss 0.8609 MAE val PAS/PAD 13.32/8.82 mmHg (16s)


In [4]:
fig, ax = plt.subplots(1, 3, figsize=(14, 3.6))
for name, res in runs.items():
    h = res['history']
    ax[0].plot([r['train_loss'] for r in h], label=name); ax[1].plot([r['val_loss'] for r in h], label=name)
    ax[2].plot([r['val_mae_pas_mmHg'] for r in h], label=name)
for a, t in zip(ax, ['train_loss (MSE estandarizado)', 'val_loss', 'val MAE PAS (mmHg)']):
    a.set_title(t); a.set_xlabel('época'); a.legend()
plt.tight_layout(); plt.savefig(ROOT / 'artifacts' / 'fig_curvas_gru.png', dpi=110)
import pandas as pd
pd.DataFrame([{'entrada': k, 'mejor_epoca': r['best_epoch'], 'val_loss': r['best_val_loss'], 'MAE_PAS': r['val_mae_pas'], 'MAE_PAD': r['val_mae_pad']}
              for k, r in runs.items()])

,entrada,mejor_epoca,val_loss,MAE_PAS,MAE_PAD
0,ppg,13,0.797275,13.199132,8.519183
1,ecg,1,0.909039,14.009351,9.330825
2,ppg+ecg,5,0.860899,13.317765,8.817842
